# QuantumBankAI - qBraid Training Notebook ⚛️
This notebook is completely self-contained and designed to be run on **qBraid** (or any Jupyter environment with Qiskit installed). 

It trains the **Quantum Support Vector Classifier (QSVC)** using a 3-qubit `ZZFeatureMap` (which consists of exactly 12 quantum gates) to detect fraudulent transactions.


In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-aer scikit-learn pandas numpy matplotlib


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer.primitives import Sampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

print("All libraries imported successfully!")


## 1. Load & Preprocess Data
For simplicity on qBraid, we will generate a synthetic dataset that mimics the properties of our fraud data. You can replace this block with `pd.read_csv("your_data.csv")` when you upload your actual dataset.


In [ ]:
# Generate synthetic fraud-like data (or replace with your actual CSV)
X, y = make_classification(
    n_samples=500, 
    n_features=10, 
    n_informative=5, 
    n_classes=2, 
    weights=[0.8, 0.2], # 20% fraud imbalance
    random_state=42
)

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA to reduce to 3 dimensions (so we can use our 3-qubit, 12-gate circuit)
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")


## 2. Define the Quantum Circuit (ZZFeatureMap)
We will use a 3-qubit `ZZFeatureMap` with 1 repetition and linear entanglement. As we discovered, this mathematically decomposes into exactly **12 quantum gates**.


In [ ]:
# Define the 3-qubit feature map
feature_map = ZZFeatureMap(feature_dimension=3, reps=1, entanglement='linear')

# Visualize the 12 decomposed gates
fig = feature_map.decompose().draw(output='mpl', style={'backgroundcolor': '#ffffff'})
plt.show()


## 3. Train the Quantum Support Vector Classifier (QSVC)
We use the `FidelityQuantumKernel` to evaluate the inner products of our quantum states, and then pass that kernel to the QSVC.


In [ ]:
print("Initializing Quantum Kernel...")
# The Sampler primitive executes the circuits on the local AerSimulator by default
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

print("Training QSVC... (This might take a moment)")
start_time = time.time()
qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train, y_train)
end_time = time.time()

print(f"Training completed in {end_time - start_time:.2f} seconds.")


## 4. Evaluation
Let's see how well our quantum model performs on the test set.


In [ ]:
print("Predicting on test set...")
y_pred = qsvc.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Display Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.matshow(cm, cmap=plt.cm.Blues)
plt.title('Quantum Confusion Matrix')
fig.colorbar(cax)
ax.set_xticklabels([''] + ['Normal', 'Fraud'])
ax.set_yticklabels([''] + ['Normal', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()
